# Python para procesamiento de imágenes
## Dr. Jesús Emmanuel Solís Pérez
### jsolisp@unam.mx

---

# DESCRIPCIÓN GENERAL: 
Python para procesamiento de imágenes es un taller intensivo diseñado para entender las matemáticas de la visión computacional. En este curso se explorará cómo las imágenes son, matrices numéricas y cómo el álgebra lineal y el cálculo se convierten en herramientas para manipular la realidad visual. Los participantes, también, aprenderán a segmentar objetos utilizando espacios de color avanzados, a aplicar operadores matemáticos de convolución para detectar bordes y utilizar geometría analítica para que una computadora identifique líneas en una escena.

# CONTENIDO DEL CURSO
## TEMARIO
1. Procesamiento de imágenes
    * Carga de imágenes, indexación con NumPy.
    * Operaciones matemáticas básicas.
    * Segmentación por color en HSV.
2. Filtros de suavizado y derivativos
    * Convolución matemática.
    * Filtros de suavizado.
    * Detección de bordes.
3. Geometría y detección de líneas
    * Transformada de Hough.
    * Proyecto práctico. Detección de las líneas de una carretera.

## HABILIDADES QUE APRENDERÁ
1. Interpretación de imágenes digitales como arreglos bidimensionales y tridimensionales con la librería NumPy.
2. Manipulación directa de pixeles.
3. Conversión matemática entre el espacio de color RGB y el espacio HSV.
4. Creación de máscaras binarias mediante umbrales
5. Implementación del algoritmo de Canny.
6. Uso de la Transformada de Hough.

---

In [1]:
import cv2
import numpy as np

In [2]:
def adquisicion_online():
    """
    Función para la adquisición y visualización de video en tiempo real 
    utilizando la cámara web predeterminada.
    """
    
    # Inicializar la captura de video desde la cámara predeterminada (índice 0)
    cap = cv2.VideoCapture(0)

    # Verificar si la cámara se abrió correctamente
    if not cap.isOpened():
        print("Error: No se pudo abrir la cámara.")
        return

    # Bucle principal para la lectura continua de fotogramas
    while True:
        # Capturar el fotograma actual (ret es un booleano que indica éxito, frame es la imagen)
        ret, frame = cap.read()

        # Comprobar si el fotograma se leyó correctamente
        if not ret:
            print("Error: No se pudo leer el frame.")
            break

        # Mostrar el fotograma capturado en una ventana titulada 'Adquisición Online'
        cv2.imshow('Adquisición Online', frame)

        # Esperar 1 milisegundo y comprobar si se presionó la tecla 'q' para salir
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Liberar los recursos de la cámara
    cap.release()
    
    # Cerrar todas las ventanas abiertas por OpenCV
    cv2.destroyAllWindows()

In [3]:
def nada(x):
    """
    Función de devolución de llamada (callback) vacía requerida por OpenCV 
    al crear trackbars. 
    
    Parámetros:
        x (int): Valor actual del trackbar (no se utiliza en esta función vacía,
                 pero es obligatorio que el callback lo reciba).
    """
    pass

def deteccion_bordes_canny():
    """
    Función principal que realiza la captura de video desde la cámara web
    y aplica el algoritmo de detección de bordes de Canny en tiempo real,
    permitiendo ajustar los umbrales dinámicamente mediante trackbars.
    """
    
    # -------------------------------------------------------------------------
    # 1. Inicialización y Configuración
    # -------------------------------------------------------------------------
    
    # Inicializar la captura de video (0 indica el índice de la cámara web predeterminada)
    cap = cv2.VideoCapture(0)

    # Validar si la cámara se abrió correctamente
    if not cap.isOpened():
        print("Error: No se pudo abrir la cámara.")
        return

    # Crear una ventana redimensionable para mostrar los controles y el resultado
    cv2.namedWindow('Deteccion de Bordes Canny (Tiempo Real)')

    # Crear trackbars (deslizadores) interactivos vinculados a la ventana creada
    # Sintaxis: createTrackbar(nombre, ventana, valor_inicial, valor_maximo, funcion_callback)
    cv2.createTrackbar('Umbral 1', 'Deteccion de Bordes Canny (Tiempo Real)', 50, 300, nada)
    cv2.createTrackbar('Umbral 2', 'Deteccion de Bordes Canny (Tiempo Real)', 150, 300, nada)

    print("Presiona la tecla 'ESC' para salir.")

    # -------------------------------------------------------------------------
    # 2. Bucle Principal de Procesamiento por Fotogramas
    # -------------------------------------------------------------------------
    while True:
        # Capturar el fotograma actual de la cámara
        # ret: booleano que indica si la lectura fue exitosa
        # frame: matriz (numpy array) con la imagen en formato BGR
        ret, frame = cap.read()
        
        # Verificar si el fotograma se recibió correctamente
        if not ret:
            print("Error: No se pudo recibir el fotograma.")
            break

        # Voltear la imagen horizontalmente (efecto espejo) para una interacción más natural
        frame = cv2.flip(frame, 1)

        # Paso 1: Convertir la imagen de BGR a escala de grises 
        # (El algoritmo Canny opera internamente sobre imágenes de un solo canal)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Paso 2: Aplicar un filtro Gaussiano para reducir el ruido de alta frecuencia
        # Esto previene la detección de falsos bordes provocados por el ruido del sensor
        blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)

        # Paso 3: Obtener dinámicamente los valores actuales configurados en los trackbars
        t1 = cv2.getTrackbarPos('Umbral 1', 'Deteccion de Bordes Canny (Tiempo Real)')
        t2 = cv2.getTrackbarPos('Umbral 2', 'Deteccion de Bordes Canny (Tiempo Real)')

        # Paso 4: Aplicar el algoritmo de detección de bordes de Canny
        # t1: Umbral inferior para la histéresis
        # t2: Umbral superior para la histéresis
        edges = cv2.Canny(blurred, t1, t2)

        # Paso 5: Convertir la imagen de bordes (monocromática) a formato BGR (3 canales)
        # Esto es necesario para poder concatenarla visualmente junto al video original a color
        edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

        # Paso F: Combinar horizontalmente el video original y la máscara de bordes
        # Nota: Ambas imágenes deben coincidir exactamente en resolución vertical (altura)
        combined = np.hstack((frame, edges_bgr))

        # Mostrar la imagen combinada resultante en la ventana
        cv2.imshow('Deteccion de Bordes Canny (Tiempo Real)', combined)

        # -------------------------------------------------------------------------
        # 3. Control de Salida
        # -------------------------------------------------------------------------
        # Esperar 1 milisegundo por una pulsación de tecla. 
        # Se aplica una máscara (& 0xFF) para asegurar la compatibilidad de códigos ASCII de 64 bits.
        # El código 27 corresponde a la tecla 'ESC' (Escape).
        if cv2.waitKey(1) & 0xFF == 27:
            break

    # -------------------------------------------------------------------------
    # 4. Liberación de Recursos
    # -------------------------------------------------------------------------
    # Liberar el objeto de captura de video para liberar la cámara del sistema operativo
    cap.release()
    
    # Destruir y cerrar todas las ventanas GUI creadas por OpenCV
    cv2.destroyAllWindows()

In [ ]:
# Descomentar las siguientes líneas si se ejecuta en un entorno Linux con problemas de compatibilidad con Qt.
# import os
# os.environ["QT_QPA_PLATFORM"] = "xcb"

# adquisicion_online() # Adquisicion de video en tiempo real
deteccion_bordes_canny() # Detección de bordes en tiempo real usando Canny

Presiona la tecla 'ESC' para salir.
